# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'projects / portfolio',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects / portfolio',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'portfolio / avatar', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'curriculum vitae', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'forum page', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.2
Updated
3 days ago
•
67.1k
•
2.48k
baidu/Unlimited-OCR
Updated
1 day ago
•
70.7k
•
891
yuxinlu1/gemma-4-12B-agentic-fable5-composer2.5-v2-3.5x-tau2-GGUF
Updated
7 days ago
•
165k
•
614
yuxinlu1/gemma-4-12B-coder-fable5-composer2.5-v1-GGUF
Updated
7 days ago
•
496k
•
2.36k
emper

In [22]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5.2\nUpdated\n3 days ago\n•\n67.1k\n•\n2.48k\nbaidu/Unlimited-OCR\nUpdated\n1 day ago\n•\n70.7k\n•\n891\nyuxinlu1/

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is a vibrant AI community and collaboration platform dedicated to building the future of machine learning (ML). At its core, Hugging Face empowers the global ML community — from engineers and scientists to end users — to explore, share, and experiment with open-source models, datasets, and applications.  

The platform hosts over 2 million models and more than 500,000 datasets, creating a thriving ecosystem where innovation and ethical AI development come together. Hugging Face is recognized as a central hub for open collaboration in machine learning, championing transparency and accessibility.

---

## What Hugging Face Offers

- **Models:** Explore an extensive library of over 2 million trained machine learning models covering numerous applications including natural language processing, computer vision, and more.
- **Datasets:** Browse over half a million datasets that power research and production ML systems.
- **Spaces:** Deploy and share ML-powered web applications within a collaborative environment.
- **Buckets:** Scalable storage solutions for managing datasets and models.
- **HuggingChat & Tools:** Conversational AI platforms and advanced ML toolkits to experiment and build.
- **Enterprise Solutions:** Tailored offerings including enterprise support, inference endpoints, and advanced integration tools designed for teams and businesses.

---

## Company Culture  
Hugging Face fosters a collaborative, open, and fast-growing community that embraces innovation and ethical AI development. The company is driven by a talented science team exploring cutting-edge technology and committed to supporting the next generation of ML engineers and researchers.  

The culture values transparency, openness, and accessibility, encouraging users worldwide to learn, contribute, and share breakthroughs together in a supportive environment. Hugging Face actively nurtures its community through forums, Discord channels, blogs, and shared resources.

---

## Customers & Community  
Hugging Face serves a diverse audience ranging from individual researchers and hobbyists to large enterprises. Its user base is deeply engaged in:

- Research and academic institutions pushing the frontiers of AI.
- Tech companies integrating state-of-the-art ML models into products.
- Startups building innovative AI applications.
- Developers eager to experiment and test new AI capabilities.
- Ethical AI advocates focused on open and responsible AI development.

---

## Careers & Opportunities  
Hugging Face is continuously expanding its talented team of engineers, researchers, and community managers. They welcome passionate individuals eager to contribute to an open-source AI future and work at the edge of machine learning technology.  

Opportunities include roles in software engineering, research science, developer relations, and more. Joining Hugging Face means becoming part of a global community shaping the AI tools of tomorrow.  

---

## Connect with Hugging Face  
Stay engaged with the community through:  

- [Discord](https://discord.gg/huggingface)  
- [GitHub](https://github.com/huggingface)  
- [Forum](https://discuss.huggingface.co)  
- [Blog & Daily Papers](https://huggingface.co/blog)  

Discover more and start collaborating today at **huggingface.co**  

---

### Brand Colors  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

---

Hugging Face — The AI community building the future. Join us to explore, collaborate, and innovate together!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## Who We Are  
**Hugging Face** is the AI community building the future. We serve as the central hub and collaboration platform for the global machine learning community. Our mission is to accelerate and democratize access to machine learning by enabling creators, developers, researchers, and enterprises to share and build on models, datasets, and AI applications seamlessly.

---

## Our Platform  

- **2M+ Models:** Access and collaborate on over two million machine learning models spanning a wide array of tasks and domains.  
- **500k+ Datasets:** Explore vast, diverse datasets to train, validate, and improve your AI systems.  
- **1M+ Applications & Spaces:** Engage with thousands of AI-powered applications, share interactive demos, and innovate collaboratively.  
- **Tools & Features:**  
  - **Spaces:** Host and run machine learning applications easily in a shared environment.  
  - **Buckets:** Cloud storage designed for scalable AI workloads.  
  - **HuggingChat:** Conversational AI tools powered by the latest models.
  - **Inference Endpoints & Providers:** Enterprise-ready solutions for scalable model inference and deployment.

---

## Who Uses Hugging Face  
Our platform caters to a diverse set of users:  
- **Researchers & Academics:** Accelerate research with open datasets and state-of-the-art models.  
- **Developers & Engineers:** Build, test, and deploy machine learning applications quickly.  
- **Enterprises:** Harness AI at scale with enterprise support, dedicated inference solutions, and secure storage.  
- **AI Enthusiasts & Students:** Learn and contribute to a vibrant open-source community via forums and collaborative tools.  

---

## Company Culture  
- **Community-Driven:** Hugging Face thrives on open collaboration, inclusivity, and shared progress within the AI and ML community worldwide.  
- **Open Source Focused:** We champion transparency and open source innovation, making advanced AI accessible to all.  
- **Learning & Growth:** Through active forums, Discord channels, blogs, and daily research paper discussions, we foster continuous learning and cutting-edge development.  
- **Innovation At Core:** We encourage experimentation, creativity, and pushing boundaries in machine learning technologies.

---

## Careers at Hugging Face  
Join a forward-thinking team dedicated to shaping the future of AI:  
- **Roles in Machine Learning, Engineering, Product, and Community Management** open regularly.  
- Work alongside industry experts and passionate contributors in a global, remote-friendly environment.  
- Access to industry-leading resources and opportunities for professional growth.  
- Help build scalable, impactful AI products used by millions around the world.

Explore current openings and apply via our website.

---

## Connect with Us  
- **Website:** [huggingface.co](https://huggingface.co)  
- **Community:** Forums and Discord for collaboration and support  
- **GitHub:** Open-source repositories and contributions  
- **Blog & Learning:** Daily posts, papers, and tutorials to keep you at the forefront of AI

---

**Hugging Face — Where the Machine Learning Community Creates the Future Together**

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face: The AI Playground Where the Future Hangs Out

---

## Who Are We?

Hugging Face isn’t just a company; it’s a buzzing hive of AI enthusiasts, nerds, dreamers, and code poets crafting the future with machine learning magic. Think of us as the friendly neighborhood clubhouse for AI – where models, datasets, apps, and innovative minds meet, mingle, and make things happen.

- **Mission:** The AI community building the future.
- **Home Base:** The go-to platform for collaboration on ML models, datasets, and applications.
- **Scale:** Over 2 million models and 500,000+ datasets to inspire your next breakthrough.

---

## What Do We Do?

At Hugging Face, we blend brains and bytes, creating a playground that hosts:

- **Models:** Browse and collaborate on AI models that power everything from text generation to image editing — updated more often than your favorite memes.
- **Datasets:** Dig into a treasure trove of over half a million datasets. If data is the new oil, consider us your refinery.
- **Spaces:** Interactive AI apps and experiments that actually run—in your browser! Imagine chatting with AI models, editing images with a few clicks, or even generating videos from single images. Yep, we’ve got it.
- **Enterprise Solutions:** Custom AI support, inference endpoints, and scalable storage buckets for businesses ready to embrace AI with open arms (and CPUs).
- **HuggingChat:** Our own AI chat platform where you can rub digital elbows with cutting-edge conversational AI.

---

## Our Culture: Where AI Meets Community

- **Collaboration is King:** We’re a community-first tribe, fostering open-source vibes so everyone—from solo data scientists to global organizations—can create and share.
- **Innovation on Steroids:** Our team thrives on fresh ideas, speedy updates, and cutting-edge research distilled into practical tools.
- **Fun Meets Function:** From quirky AI agent names like “OpenMythos” to the friendly “Pro Realism Edit Studio,” we don’t just build AI — we _enjoy_ it.
- **Support Like No Other:** Whether you’re an indie developer or an enterprise powerhouse, we’ve got Huggies (er… support) tailored just for you.

---

## Who’s Hugging Face For?

- **AI Researchers & Developers:** Dive into a platform packed with ready-to-use and customizable models — updated so regularly they'll make your morning coffee jealous.
- **Businesses:** Need enterprise-level AI with hassle-free hosting and robust inference services? We speak your language.
- **Students & Educators:** Jump in on the latest AI tools, datasets, and papers to learn, experiment, and teach the art of the possible.
- **AI Curious:** Whether you’re a hobbyist or a futurist, our Spaces are your sandbox for playing with AI wonders without messing with code.

---

## Join Our Team! Careers at Hugging Face

Got a passion for AI, open-source, and shaking up the tech world? We’re hiring bold thinkers, builders, and boundary-pushers who:

- Love collaborating in a fast-moving, mission-driven environment.
- Want to impact millions of users and developers worldwide.
- Enjoy blending cutting-edge research with practical, creative applications.
- Are ready to take machine learning from “hmm” to “heck yeah!” every day.

Perks include working with the brightest minds in AI, a culture supportive of innovation (and sometimes memes), and the chance to sculpt the very future of machine learning.

---

## Why Hugging Face? Because We Make AI Hug-Worthy

From hosting millions of models to cultivating a community that’s as passionate as a TED Talk audience after espresso, Hugging Face is the place to build, share, and push the boundaries of AI. 

Ready to be part of something truly groundbreaking? Pop on over to our [website](https://huggingface.co), explore millions of models, chat with AI, or even upload your own masterpiece.

*Hugging Face: Where the machines learn and the humans play.*

---

## Fun Fast Facts:

- Colors: Bright yellow (#FFD21E) — because AI should shine as bright as your code.
- Mascot: An adorable, cartoonish smiley face with hugging arms... because hugs > bugs.
- Nearly 400 AI-powered apps running live—chatbots, editing studios, video generators, you name it.

---

**Explore. Collaborate. Innovate.**  
Get ready for the AI hug of your life! 🤗

---

### Contacts and Socials:

- Community on Discord & Forum  
- Labs on GitHub  
- Latest insights on the Blog & Daily Papers  
- Enterprise Support & PRO features for power users  

---

*Hugging Face* — _hugging your AI dreams to life since forever._

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>